In [ ]:
# Installazione dipendenze (eseguire solo al primo avvio)
# %pip install pandas matplotlib seaborn anthropic lxml --quiet

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

# ── Percorsi ──────────────────────────────────────────────────────────────────
DATA_DIR = Path("data")
DATASET_FILE = DATA_DIR / "dataset.json"

print(f"Directory dati : {DATA_DIR.resolve()}")
print(f"File dataset   : {DATASET_FILE.resolve()}")
print(f"Esiste         : {DATASET_FILE.exists()}")

In [ ]:
def load_dataset(dataset_file: Path, data_dir: Path) -> pd.DataFrame:
    """
    Carica dataset.json e arricchisce ogni record con:
    - il testo dell'istruzione (input.txt)
    - il contenuto XML di ogni file output
    - il numero di output attesi
    """
    with open(dataset_file, encoding="utf-8") as f:
        records = json.load(f)

    enriched = []
    for rec in records:
        entry = dict(rec)

        # Legge input.txt
        input_path = Path(rec["input_file"])
        if input_path.exists():
            entry["instruction"] = input_path.read_text(encoding="utf-8").strip()
        else:
            entry["instruction"] = None

        # Legge ogni file output XML
        outputs = []
        for out_path_str in rec.get("output_files", []):
            out_path = Path(out_path_str)
            if out_path.exists():
                outputs.append(out_path.read_text(encoding="utf-8").strip())
        entry["outputs"] = outputs
        entry["n_outputs"] = len(outputs)
        entry["has_output"] = len(outputs) > 0

        # Calcola lunghezza istruzione (proxy di complessità del testo)
        entry["instruction_len"] = len(entry["instruction"]) if entry["instruction"] else 0

        # Numero di tag
        entry["n_tags"] = len(rec.get("tags", []))

        enriched.append(entry)

    df = pd.DataFrame(enriched)

    # Ordine logico delle colonne
    cols = ["id", "difficulty", "type", "n_outputs", "has_output",
            "n_tags", "instruction_len", "instruction", "tags",
            "description", "input_file", "output_files", "outputs"]
    df = df[[c for c in cols if c in df.columns]]

    return df


df = load_dataset(DATASET_FILE, DATA_DIR)
print(f"Casistiche caricate: {len(df)}")
print(f"Con output XML    : {df['has_output'].sum()}")
print(f"Senza output XML  : {(~df['has_output']).sum()} (da completare)")

In [ ]:
# Panoramica del dataset
df[["id", "difficulty", "type", "n_tags", "instruction_len", "has_output", "description"]]

In [ ]:
# Verifica un esempio completo
EXAMPLE_ID = "01"

row = df[df["id"] == EXAMPLE_ID].iloc[0]

print("=" * 60)
print(f"ID          : {row['id']}")
print(f"Difficoltà  : {row['difficulty']}")
print(f"Tipo        : {row['type']}")
print(f"Tag         : {', '.join(row['tags'])}")
print(f"Descrizione : {row['description']}")
print()
print("── ISTRUZIONE " + "-" * 45)
print(row["instruction"])
print()
if row["outputs"]:
    print("── OUTPUT XML " + "-" * 45)
    print(row["outputs"][0][:800], "..." if len(row["outputs"][0]) > 800 else "")
else:
    print("── OUTPUT XML: non ancora disponibile")

In [ ]:
# Statistiche descrittive del dataset
print("Distribuzione per difficoltà:")
print(df["difficulty"].value_counts().to_string())
print()
print("Distribuzione per tipo:")
print(df["type"].value_counts().to_string())
print()
print("Lunghezza istruzioni (caratteri):")
print(df["instruction_len"].describe().round(1).to_string())

---
## 3. ForgeAI — Il Sistema Generatore

Il sistema è composto da tre moduli:

1. **RAG** — scarica la documentazione Apache Hop, la indicizza in un vector store (ChromaDB) e recupera i chunk più rilevanti per ogni richiesta
2. **Prompt builder** — assembla system prompt + few-shot examples dal dataset + chunk RAG + istruzione utente
3. **LLM client** — invia il prompt all'API (Anthropic o OpenAI) e restituisce l'XML generato

In [ ]:
# Installazione dipendenze aggiuntive per RAG
# %pip install chromadb sentence-transformers requests beautifulsoup4 anthropic openai --quiet

In [ ]:
import os
import re
import time
import textwrap
import requests
from bs4 import BeautifulSoup
from pathlib import Path

import chromadb
from chromadb.utils import embedding_functions

# ── Configurazione ────────────────────────────────────────────────────────────
PROVIDER      = "anthropic"   # "anthropic" | "openai"
MODEL_ANTHROPIC = "claude-sonnet-4-5"
MODEL_OPENAI    = "gpt-4o-mini"
MAX_TOKENS    = 4096
TOP_K_RAG     = 4             # chunk RAG da iniettare nel prompt
FEW_SHOT_K    = 3             # esempi few-shot dal dataset

CHROMA_DIR    = Path("data/chroma_db")
COLLECTION    = "hop_docs"

# API keys — imposta come variabili d'ambiente oppure assegna direttamente
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY", "")

print(f"Provider      : {PROVIDER}")
print(f"ChromaDB dir  : {CHROMA_DIR}")
print(f"Anthropic key : {'OK' if ANTHROPIC_API_KEY else 'MANCANTE'}")
print(f"OpenAI key    : {'OK' if OPENAI_API_KEY else 'MANCANTE'}")

### 3.1 Raccolta documentazione Apache Hop

In [ ]:
# Pagine della documentazione ufficiale da indicizzare
# https://hop.apache.org/manual/latest/
HOP_DOC_URLS = [
    "https://hop.apache.org/manual/latest/pipeline/pipelines.html",
    "https://hop.apache.org/manual/latest/workflow/workflows.html",
    "https://hop.apache.org/manual/latest/pipeline/transforms/csvinput.html",
    "https://hop.apache.org/manual/latest/pipeline/transforms/tableinput.html",
    "https://hop.apache.org/manual/latest/pipeline/transforms/tableoutput.html",
    "https://hop.apache.org/manual/latest/pipeline/transforms/textfileoutput.html",
    "https://hop.apache.org/manual/latest/pipeline/transforms/filterrows.html",
    "https://hop.apache.org/manual/latest/pipeline/transforms/selectvalues.html",
    "https://hop.apache.org/manual/latest/pipeline/transforms/databaselookup.html",
    "https://hop.apache.org/manual/latest/pipeline/transforms/mergejoin.html",
    "https://hop.apache.org/manual/latest/pipeline/transforms/groupby.html",
    "https://hop.apache.org/manual/latest/pipeline/transforms/insertupdate.html",
    "https://hop.apache.org/manual/latest/pipeline/transforms/rowgenerator.html",
    "https://hop.apache.org/manual/latest/pipeline/transforms/writetolog.html",
    "https://hop.apache.org/manual/latest/workflow/actions/pipeline.html",
    "https://hop.apache.org/manual/latest/workflow/actions/fileexists.html",
    "https://hop.apache.org/manual/latest/workflow/actions/movefile.html",
    "https://hop.apache.org/manual/latest/workflow/actions/mail.html",
    "https://hop.apache.org/manual/latest/workflow/actions/abort.html",
]

def fetch_doc_page(url: str) -> str:
    """Scarica una pagina della doc e restituisce il testo pulito."""
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        # Rimuove nav, header, footer
        for tag in soup(["nav", "header", "footer", "script", "style"]):
            tag.decompose()
        main = soup.find("main") or soup.find("article") or soup.find("body")
        text = main.get_text(separator="\n", strip=True) if main else ""
        # Normalizza spazi bianchi
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text
    except Exception as e:
        print(f"  ERRORE {url}: {e}")
        return ""


def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """Divide il testo in chunk con overlap."""
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i + chunk_size])
        if len(chunk.strip()) > 50:  # scarta chunk troppo corti
            chunks.append(chunk)
        i += chunk_size - overlap
    return chunks


print(f"Pagine da scaricare: {len(HOP_DOC_URLS)}")
print("Esegui la cella successiva per avviare il download.")

In [ ]:
# Download e chunking (richiede connessione internet, ~1-2 min)
all_chunks = []
all_metadata = []

for url in HOP_DOC_URLS:
    page_name = url.split("/")[-1].replace(".html", "")
    print(f"  Scaricando {page_name}...", end=" ")
    text = fetch_doc_page(url)
    if text:
        chunks = chunk_text(text)
        all_chunks.extend(chunks)
        all_metadata.extend([{"source": url, "page": page_name}] * len(chunks))
        print(f"{len(chunks)} chunk")
    else:
        print("saltata")
    time.sleep(0.3)  # rispetta il server

print(f"\nTotale chunk raccolti: {len(all_chunks)}")

### 3.2 Indicizzazione in ChromaDB

In [ ]:
# Crea o carica il vector store ChromaDB
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# Embedding con sentence-transformers (locale, no API key)
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Ricrea la collection se esiste già (utile per aggiornamenti)
try:
    client.delete_collection(COLLECTION)
    print("Collection esistente eliminata.")
except:
    pass

collection = client.create_collection(
    name=COLLECTION,
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"}
)

# Indicizza i chunk in batch
BATCH_SIZE = 100
for i in range(0, len(all_chunks), BATCH_SIZE):
    batch_docs  = all_chunks[i:i + BATCH_SIZE]
    batch_meta  = all_metadata[i:i + BATCH_SIZE]
    batch_ids   = [f"doc_{i + j}" for j in range(len(batch_docs))]
    collection.add(documents=batch_docs, metadatas=batch_meta, ids=batch_ids)
    print(f"  Indicizzati {min(i + BATCH_SIZE, len(all_chunks))}/{len(all_chunks)} chunk")

print(f"\nChromaDB pronto — {collection.count()} documenti indicizzati.")
print(f"Persistito in: {CHROMA_DIR.resolve()}")

### 3.3 Prompt builder

In [ ]:
SYSTEM_PROMPT = """\
Sei ForgeAI, un assistente specializzato nella generazione di file Apache Hop.
Apache Hop è un framework ETL open source. I file di pipeline hanno estensione .hpl,
i file di workflow hanno estensione .hwf. Entrambi sono serializzati in XML.

Il tuo compito è trasformare istruzioni in linguaggio naturale in file XML Apache Hop
validi e completi. Segui queste regole:

1. Rispondi SOLO con l'XML, senza spiegazioni, senza markdown, senza backtick.
2. L'XML deve iniziare con <?xml version="1.0" encoding="UTF-8"?>
3. Usa i tag <pipeline> per le pipeline e <workflow> per i workflow.
4. Ogni transform/action deve avere un tag <GUI> con <xloc> e <yloc> per il layout.
5. Collega sempre tutti i transform/action con i relativi <hop> nella sezione <order>.
6. Usa nomi descrittivi in italiano per i transform/action.
7. Per le connessioni DB usa il tag <connection> con il nome del database.
"""

def select_few_shot_examples(instruction: str, df, k: int = 3) -> list[dict]:
    """
    Seleziona k esempi dal dataset con output XML disponibile.
    Strategia semplice: prende esempi con has_output=True distribuiti
    per difficoltà (easy, medium, hard).
    """
    available = df[df["has_output"] == True].copy()
    if len(available) == 0:
        return []

    selected = []
    for diff in ["easy", "medium", "hard"]:
        subset = available[available["difficulty"] == diff]
        if len(subset) > 0:
            selected.append(subset.iloc[0])
        if len(selected) >= k:
            break

    return selected


def retrieve_rag_chunks(instruction: str, collection, k: int = 4) -> str:
    """Recupera i k chunk più rilevanti dalla documentazione."""
    results = collection.query(
        query_texts=[instruction],
        n_results=k
    )
    chunks = results["documents"][0]
    sources = [m["page"] for m in results["metadatas"][0]]
    formatted = []
    for chunk, source in zip(chunks, sources):
        formatted.append(f"[Fonte: {source}]\n{chunk}")
    return "\n\n---\n\n".join(formatted)


def build_prompt(instruction: str, df, collection) -> list[dict]:
    """
    Assembla la lista di messaggi per l'API:
    - few-shot examples come coppie user/assistant
    - chunk RAG iniettati nel messaggio utente finale
    """
    messages = []

    # Few-shot examples
    examples = select_few_shot_examples(instruction, df, k=FEW_SHOT_K)
    for ex in examples:
        messages.append({"role": "user", "content": ex["instruction"]})
        messages.append({"role": "assistant", "content": ex["outputs"][0]})

    # Messaggio finale con RAG
    rag_context = retrieve_rag_chunks(instruction, collection, k=TOP_K_RAG)
    user_message = (
        f"Documentazione di riferimento:\n\n{rag_context}\n\n"
        f"---\n\n"
        f"Istruzione: {instruction}"
    )
    messages.append({"role": "user", "content": user_message})

    return messages


print("Prompt builder pronto.")

### 3.4 LLM Client

In [ ]:
def call_llm(messages: list[dict], system: str = SYSTEM_PROMPT) -> str:
    """
    Chiama l'API LLM configurata (Anthropic o OpenAI).
    Restituisce il testo generato.
    """
    if PROVIDER == "anthropic":
        import anthropic
        client_llm = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        response = client_llm.messages.create(
            model=MODEL_ANTHROPIC,
            max_tokens=MAX_TOKENS,
            system=system,
            messages=messages
        )
        return response.content[0].text.strip()

    elif PROVIDER == "openai":
        from openai import OpenAI
        client_llm = OpenAI(api_key=OPENAI_API_KEY)
        full_messages = [{"role": "system", "content": system}] + messages
        response = client_llm.chat.completions.create(
            model=MODEL_OPENAI,
            max_tokens=MAX_TOKENS,
            messages=full_messages
        )
        return response.choices[0].message.content.strip()

    else:
        raise ValueError(f"Provider non supportato: {PROVIDER}")


def generate(instruction: str) -> str:
    """
    Funzione principale di ForgeAI.
    Input:  istruzione in linguaggio naturale
    Output: XML Apache Hop generato
    """
    messages = build_prompt(instruction, df, collection)
    xml_output = call_llm(messages)
    # Rimuove eventuali backtick residui
    xml_output = re.sub(r"^```[a-z]*\n?", "", xml_output)
    xml_output = re.sub(r"\n?```$", "", xml_output)
    return xml_output.strip()


print("LLM client pronto.")
print(f"Provider attivo: {PROVIDER}")

### 3.5 Demo interattiva

In [ ]:
# Esempi di test — modifica o aggiungi le tue istruzioni
TEST_INSTRUCTIONS = [
    "Leggi la tabella ordini da PostgreSQL sales_db e scrivila in un file CSV chiamato ordini_export.csv.",
    "Leggi il file clienti.csv, filtra solo i clienti con eta maggiore di 30, e inseriscili nella tabella clienti_adulti di MySQL crm_db.",
    "Crea un workflow che controlla se esiste il file report.csv, se sì esegue la pipeline elabora_report.hpl, altrimenti scrive nel log che il file non è stato trovato.",
]

# Esegui la generazione su un'istruzione alla volta
INSTRUCTION_INDEX = 0  # cambia indice per testare le altre

instruction = TEST_INSTRUCTIONS[INSTRUCTION_INDEX]
print(f"Istruzione:\n{instruction}")
print("\nGenerazione in corso...")

result = generate(instruction)

print("\n" + "=" * 60)
print("OUTPUT XML:")
print("=" * 60)
print(result)

In [ ]:
# Salva l'output generato su file per ispezione
output_dir = Path("data/generated")
output_dir.mkdir(exist_ok=True)

ext = ".hwf" if "<workflow>" in result else ".hpl"
out_file = output_dir / f"demo_{INSTRUCTION_INDEX}{ext}"
out_file.write_text(result, encoding="utf-8")

print(f"File salvato: {out_file}")
print(f"Tipo rilevato: {'workflow' if ext == '.hwf' else 'pipeline'}")

---
*Prossima sezione → **4. Valutazione**: metriche di performance del sistema*